# Assignment 23: OpenAI & Retrieval-Augmented Generation (RAG)
**Author:** Parth Dadhaniya  
**Course:** Generative AI Engineering

This notebook demonstrates the end-to-end implementation of Retrieval-Augmented Generation (RAG) pipelines using LangChain, embeddings, ChromaDB, advanced retrievers (MMR, Multi-Query, Contextual Compression), and a conversational YouTube Content RAG Chatbot.


## Setup and Environment


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import config

print('Environment and config loaded.')


## Part 1: Getting Started with OpenAI
### Task 1: OpenAI Setup & Basic Prompt


In [ ]:
from langchain_core.messages import HumanMessage

api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-'):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
    prompt = 'Explain Retrieval-Augmented Generation (RAG) in 2 simple sentences.'
    res = llm.invoke([HumanMessage(content=prompt)])
    print('OpenAI Response:\n', res.content)
else:
    print('OPENAI_API_KEY not set. Displaying sample response:')
    print('RAG is an AI framework that retrieves relevant documents from an external knowledge base')
    print('and passes them to a language model to generate accurate, up-to-date answers.')


## Part 2: Retriever-Based RAG (Wikipedia + Vector Store)
### Task 2: Wikipedia Retriever


In [ ]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(top_k_results=2, lang='en')
query = 'Artificial Intelligence'
print('Searching Wikipedia for:', query)

try:
    docs = retriever.invoke(query)
    for i, doc in enumerate(docs, 1):
        print(f'[{i}] Title:', doc.metadata.get('title'))
        print('    Summary:', doc.page_content[:180], '...\n')
except Exception as e:
    print('Wikipedia retrieval error:', e)


### Task 3: Vector Store Retriever


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# load and chunk document
docs = TextLoader('data/knowledge_base.txt', encoding='utf-8').load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
db = Chroma.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={'k': 2})

query = 'How does RAG solve hallucinations?'
results = retriever.invoke(query)
print('Query:', query)
for i, d in enumerate(results, 1):
    print(f'\n[Result {i}]:\n{d.page_content.strip()}')


## Part 3: Advanced Retrieval Strategies
### Task 4: Maximal Marginal Relevance (MMR) Retriever


In [ ]:
query = 'Vector databases ChromaDB and FAISS similarity search'
print('Query:', query)

sim_retriever = db.as_retriever(search_type='similarity', search_kwargs={'k': 3})
mmr_retriever = db.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5})

print('\n1. Normal Similarity Search:')
for i, d in enumerate(sim_retriever.invoke(query), 1):
    print(f'[{i}] {d.page_content.replace(chr(10), " ")[:100]}...')

print('\n2. MMR Diverse Search:')
for i, d in enumerate(mmr_retriever.invoke(query), 1):
    print(f'[{i}] {d.page_content.replace(chr(10), " ")[:100]}...')


### Task 5: Multi-Query Retriever


In [ ]:
try:
    from langchain_classic.retrievers.multi_query import MultiQueryRetriever
except ImportError:
    from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.language_models.fake import FakeListLLM

llm_mq = FakeListLLM(responses=[
    '1. What are vector databases and embeddings in RAG?\n'
    '2. How do vector stores improve retrieval in language models?\n'
    '3. Explain embeddings and vector search.'
])

mq_retriever = MultiQueryRetriever.from_llm(retriever=db.as_retriever(search_kwargs={'k': 2}), llm=llm_mq)
query = 'How do vector stores and embeddings assist RAG systems?'

single_docs = db.as_retriever(search_kwargs={'k': 2}).invoke(query)
mq_docs = mq_retriever.invoke(query)

print(f'Single Query: {len(single_docs)} chunks')
print(f'Multi-Query:  {len(mq_docs)} unique chunks')


### Task 6: Contextual Compression Retriever


In [ ]:
try:
    from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import EmbeddingsFilter
except ImportError:
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import EmbeddingsFilter

compressor = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.35)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=db.as_retriever(search_kwargs={'k': 5})
)

query = 'How does FAISS optimize fast vector search?'
raw = db.as_retriever(search_kwargs={'k': 5}).invoke(query)
comp = compression_retriever.invoke(query)

print(f'Before Compression: {len(raw)} chunks')
print(f'After Compression:  {len(comp)} chunks')


## Part 4: YouTube Content RAG Chatbot
### Task 7: Load YouTube Video Content


In [ ]:
from langchain_community.document_loaders import YoutubeLoader, TextLoader

video_url = 'https://www.youtube.com/watch?v=kCc8FmEb1nY'
try:
    yt_docs = YoutubeLoader.from_youtube_url(video_url, add_video_info=False).load()
    print('Live transcript loaded. Length:', len(yt_docs[0].page_content))
except Exception:
    yt_docs = TextLoader('data/youtube_transcript.txt', encoding='utf-8').load()
    print('Local backup transcript loaded. Length:', len(yt_docs[0].page_content))

yt_splitter = RecursiveCharacterTextSplitter(chunk_size=280, chunk_overlap=30)
yt_chunks = yt_splitter.split_documents(yt_docs)
print('Total chunks:', len(yt_chunks))


### Task 8: YouTube Embeddings & Vector Store


In [ ]:
yt_db = Chroma.from_documents(yt_chunks, embeddings, persist_directory='./chroma_youtube')
test_q = 'What are the two files that make up an LLM?'
match = yt_db.as_retriever(search_kwargs={'k': 1}).invoke(test_q)
print('Test Match:\n', match[0].page_content.strip())


### Task 9: YouTube Content RAG Chatbot


In [ ]:
from task9_youtube_rag_chatbot import YouTubeChatbot

bot = YouTubeChatbot()
q = 'What are the two files that make up an LLM?'
ans, _ = bot.ask(q)
print('Q:', q)
print('A:', ans)


### Task 10: Testing & Evaluation


In [ ]:
from task10_testing_evaluation import test_questions

for i, q in enumerate(test_questions, 1):
    ans, _ = bot.ask(q)
    print(f'Q{i}: {q}')
    print(f'A{i}: {ans}\n')


## Part 5: Observations & Conceptual Questions
### Task 11: Conceptual Questions & Answers


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'task11_observations.py'])
